<a href="https://colab.research.google.com/github/usami0jp/ai/blob/main/Transformer2ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#      　トランスフォーマー

import random
import math
from typing import List

import numpy as np
import torch
import matplotlib.pyplot as plt


try:
    torch.set_num_threads(2)
except Exception:
    pass


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


set_seed(42)
device = get_device()
print("device =", device)

#＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝
# ===== トークナイザー tokenizer =====
#＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝


class Tokenizer:
    PAD = "<pad>"

    def __init__(self, training_text: str):
        self.dictionary = {}
        self.reverse_dictionary = {}

        self._add_to_dict(self.PAD)

        # 学習テキストに出る文字を登録

        for ch in sorted(set(training_text)):
            self._add_to_dict(ch)

    def _add_to_dict(self, character: str):
        if character not in self.dictionary:
            token_id = len(self.dictionary)
            self.dictionary[character] = token_id
            self.reverse_dictionary[token_id] = character

    def tokenize(self, text: str) -> List[int]:
        unknown = sorted(set(text) - set(self.dictionary.keys()))
        if unknown:
            raise ValueError(
                f"Tokenizerにない文字があります: {unknown}\n"
                f"training_dataにその文字を含めるか、promptを変えてください。"
            )
        return [self.dictionary[c] for c in text]

    def character_to_token(self, character: str) -> int:
        return self.dictionary[character]

    def token_to_character(self, token: int) -> str:
        return self.reverse_dictionary[int(token)]

    def decode(self, tokens: List[int], skip_special: bool = True) -> str:
        chars = []
        for t in tokens:
            ch = self.token_to_character(t)
            if skip_special and ch == self.PAD:
                continue
            chars.append(ch)
        return "".join(chars)

    def size(self) -> int:
        return len(self.dictionary)


# ===== Transformerの部品 =====

class TokenEmbedding(torch.nn.Module):
    """
    token番号をベクトルに変換する層。
    入力:  (batch_size, sequence_length)
    出力:  (batch_size, sequence_length, d_model)
    """
    def __init__(self, d_model: int, number_of_tokens: int):
        super().__init__()
        self.embedding_layer = torch.nn.Embedding(
            num_embeddings=number_of_tokens,
            embedding_dim=d_model
        )

    def forward(self, x):
        return self.embedding_layer(x)


class PositionalEncoding(torch.nn.Module):
    """
    TransformerはRNNではないので、そのままだと語順が分からない
    そこで、sin/cosで作った位置ベクトルをtoken embeddingに足す。
    """
    def __init__(self, d_model: int, max_sequence_length: int):
        super().__init__()
        self.d_model = d_model
        self.max_sequence_length = max_sequence_length
        self.register_buffer("positional_encoding", self.create_positional_encoding())

    def create_positional_encoding(self):
        positional_encoding = np.zeros((self.max_sequence_length, self.d_model), dtype=np.float32)

        for pos in range(self.max_sequence_length):
            for i in range(0, self.d_model, 2):
                positional_encoding[pos, i] = np.sin(pos / (10000 ** (i / self.d_model)))
                if i + 1 < self.d_model:
                    positional_encoding[pos, i + 1] = np.cos(pos / (10000 ** (i / self.d_model)))

        return torch.from_numpy(positional_encoding).float()

    def forward(self, x):
        # x: (batch_size, sequence_length, d_model)
        seq_len = x.size(1)
        return x + self.positional_encoding[:seq_len, :]

#＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝
# Masked Self Attention  マスクト　セルフ　アテンション
#＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝

class MaskedSelfAttention(torch.nn.Module):

    def __init__(self, embedding_dimension: int, head_dimension: int):
        super().__init__()
        self.embedding_dimension = embedding_dimension
        self.head_dimension = head_dimension

        self.query_layer = torch.nn.Linear(embedding_dimension, head_dimension)
        self.key_layer = torch.nn.Linear(embedding_dimension, head_dimension)
        self.value_layer = torch.nn.Linear(embedding_dimension, head_dimension)

    def forward(self, x, mask):
        # x:    (batch_size, sequence_length, embedding_dimension)
        # mask: (batch_size, sequence_length)  1=有効token, 0=pad
        query = self.query_layer(x)
        key = self.key_layer(x)
        value = self.value_layer(x)

        attention_weights = torch.matmul(query, key.transpose(-2, -1))
        attention_weights = attention_weights / math.sqrt(self.head_dimension)

        batch_size, seq_length, _ = attention_weights.shape

        # padding mask: pad tokenを見ない
        padding_mask = mask[:, None, :].bool()  # (batch_size, 1, sequence_length)

        # causal mask:
        causal_mask = torch.tril(
            torch.ones(seq_length, seq_length, device=x.device, dtype=torch.bool)
        )[None, :, :]  # (1, sequence_length, sequence_length)

        combined_mask = padding_mask & causal_mask
        attention_weights = attention_weights.masked_fill(~combined_mask, -1e9)

        attention_scores = torch.softmax(attention_weights, dim=-1)
        return torch.bmm(attention_scores, value)


#＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝
# Masked Multi-head Self Attention  マスクト　マルチヘッド　セルフ　アテンション
#＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝

class MaskedMultiHeadedSelfAttention(torch.nn.Module):
    """
    複数のattention headを並列に動かし、最後にconcatして線形変換する。
    """
    def __init__(self, embedding_dimension: int, number_of_heads: int):
        super().__init__()
        if embedding_dimension % number_of_heads != 0:
            raise ValueError("embedding_dimension は number_of_heads で割り切れる必要があります。")

        self.embedding_dimension = embedding_dimension
        self.number_of_heads = number_of_heads
        self.head_dimension = embedding_dimension // number_of_heads

        self.self_attentions = torch.nn.ModuleList([
            MaskedSelfAttention(embedding_dimension, self.head_dimension)
            for _ in range(number_of_heads)
        ])

        self.output_layer = torch.nn.Linear(
            number_of_heads * self.head_dimension,
            embedding_dimension
        )

    def forward(self, x, mask):
        head_outputs = [self_attention(x, mask) for self_attention in self.self_attentions]
        concatenated = torch.cat(head_outputs, dim=2)
        return self.output_layer(concatenated)


class FeedForward(torch.nn.Module):
    """
    各位置ごとに独立にかかる全結合ネットワーク。
    """
    def __init__(self, embedding_dimension: int, feed_forward_dimension: int):
        super().__init__()
        self.linear_1 = torch.nn.Linear(embedding_dimension, feed_forward_dimension)
        self.linear_2 = torch.nn.Linear(feed_forward_dimension, embedding_dimension)

    def forward(self, x):
        return self.linear_2(torch.relu(self.linear_1(x)))


class DecoderLayer(torch.nn.Module):
    """
    GPT型Transformerの1ブロック。
    LayerNorm → MultiHeadAttention → 残差接続
    LayerNorm → FeedForward       → 残差接続
    """
    def __init__(
        self,
        embedding_dimension: int,
        number_of_heads: int,
        feed_forward_dimension: int,
        dropout_rate: float
    ):
        super().__init__()

        self.multi_headed_self_attention = MaskedMultiHeadedSelfAttention(
            embedding_dimension, number_of_heads
        )
        self.feed_forward = FeedForward(embedding_dimension, feed_forward_dimension)

        self.layer_normalization_1 = torch.nn.LayerNorm(embedding_dimension)
        self.layer_normalization_2 = torch.nn.LayerNorm(embedding_dimension)

        self.dropout = torch.nn.Dropout(dropout_rate)

    def forward(self, x, mask):
        normalized_x = self.layer_normalization_1(x)
        attention_output = self.multi_headed_self_attention(normalized_x, mask)
        x = x + self.dropout(attention_output)

        normalized_x = self.layer_normalization_2(x)
        feed_forward_output = self.feed_forward(normalized_x)
        x = x + self.dropout(feed_forward_output)

        return x


class DecoderStack(torch.nn.Module):
    """
    DecoderLayerを何層も積む。
    """
    def __init__(
        self,
        embedding_dimension: int,
        number_of_layers: int,
        number_of_heads: int,
        feed_forward_dimension: int,
        dropout_rate: float
    ):
        super().__init__()
        self.decoder_layers = torch.nn.ModuleList([
            DecoderLayer(
                embedding_dimension=embedding_dimension,
                number_of_heads=number_of_heads,
                feed_forward_dimension=feed_forward_dimension,
                dropout_rate=dropout_rate
            )
            for _ in range(number_of_layers)
        ])

    def forward(self, x, mask):
        for decoder_layer in self.decoder_layers:
            x = decoder_layer(x, mask)
        return x


class LMHead(torch.nn.Module):
    """
    各位置の隠れベクトルから、次のtoken候補のlogitsを出す層。
    """
    def __init__(self, embedding_dimension: int, number_of_tokens: int):
        super().__init__()
        self.linear = torch.nn.Linear(embedding_dimension, number_of_tokens)

    def forward(self, x):
        return self.linear(x)


class LanguageModel(torch.nn.Module):
    """
    TokenEmbedding + PositionalEncoding + DecoderStack + LMHead
    """
    def __init__(
        self,
        number_of_tokens: int,
        max_sequence_length: int = 64,
        embedding_dimension: int = 64,
        number_of_layers: int = 1,
        number_of_heads: int = 4,
        feed_forward_dimension: int = None,
        dropout_rate: float = 0.1
    ):
        super().__init__()

        if feed_forward_dimension is None:
            feed_forward_dimension = embedding_dimension * 4

        self.number_of_tokens = number_of_tokens
        self.max_sequence_length = max_sequence_length
        self.embedding_dimension = embedding_dimension
        self.number_of_layers = number_of_layers
        self.number_of_heads = number_of_heads
        self.feed_forward_dimension = feed_forward_dimension
        self.dropout_rate = dropout_rate

        self.token_embedding = TokenEmbedding(embedding_dimension, number_of_tokens)
        self.positional_encoding = PositionalEncoding(embedding_dimension, max_sequence_length)
        self.layer_normalization = torch.nn.LayerNorm(embedding_dimension)

        self.decoder = DecoderStack(
            embedding_dimension=embedding_dimension,
            number_of_layers=number_of_layers,
            number_of_heads=number_of_heads,
            feed_forward_dimension=feed_forward_dimension,
            dropout_rate=dropout_rate
        )

        self.lm_head = LMHead(embedding_dimension, number_of_tokens)

    def forward(self, x, mask):
        token_embeddings = self.token_embedding(x)
        x = self.positional_encoding(token_embeddings)
        x = self.layer_normalization(x)
        x = self.decoder(x, mask)
        logits = self.lm_head(x)
        return logits

    def save_checkpoint(self, path: str):
        torch.save({
            "number_of_tokens": self.number_of_tokens,
            "max_sequence_length": self.max_sequence_length,
            "embedding_dimension": self.embedding_dimension,
            "number_of_layers": self.number_of_layers,
            "number_of_heads": self.number_of_heads,
            "feed_forward_dimension": self.feed_forward_dimension,
            "dropout_rate": self.dropout_rate,
            "model_state_dict": self.state_dict()
        }, path)
        print("saved:", path)


# ===== 学習と生成 =====

class AutoregressiveWrapper(torch.nn.Module):
    """
    x[:, :-1] から x[:, 1:] を予測するように包む。
    """
    def __init__(self, gpt_model: LanguageModel):
        super().__init__()
        self.model = gpt_model
        self.max_sequence_length = self.model.max_sequence_length

    def forward(self, x, mask):
        inp = x[:, :-1]
        target = x[:, 1:]
        mask = mask[:, :-1]
        output = self.model(inp, mask)
        return output, target

    def next_token_probabilities(self, x, mask, temperature: float = 1.0):
        logits = self.model(x, mask)[:, -1, :]
        if temperature != 1.0:
            logits = logits / temperature
        return torch.softmax(logits, dim=-1)

    def save_checkpoint(self, path: str):
        self.model.save_checkpoint(path)


def create_training_sequences(max_sequence_length: int, tokenized_training_data: List[int]):
    """
    max_sequence_length + 1 の長さの系列を作る。
    最後の1文字が教師信号になる。
    """
    sequences = []
    for i in range(0, len(tokenized_training_data) - max_sequence_length):
        sequences.append(tokenized_training_data[i:i + max_sequence_length + 1])
    return sequences


def tokenize_and_pad_training_data(max_sequence_length: int, tokenizer: Tokenizer, training_data: str):
    tokenized = tokenizer.tokenize(training_data)

    # 最初の方の系列も学習できるよう、左側にpadを足す。
    pad_id = tokenizer.character_to_token(Tokenizer.PAD)
    tokenized = [pad_id] * max_sequence_length + tokenized
    return tokenized


class Trainer:
    def __init__(self, model: AutoregressiveWrapper, tokenizer: Tokenizer, optimizer=None):
        self.model = model
        self.tokenizer = tokenizer

        if optimizer is None:
            self.optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
        else:
            self.optimizer = optimizer

        pad_id = tokenizer.character_to_token(Tokenizer.PAD)
        self.loss_function = torch.nn.CrossEntropyLoss(ignore_index=pad_id)

    def train(self, data: List[List[int]], epochs: int, batch_size: int):
        loss_per_epoch = []

        for epoch in range(1, epochs + 1):
            random.shuffle(data)
            losses = []

            for i in range(0, len(data), batch_size):
                batch = data[i:i + batch_size]
                sequence_tensor = torch.tensor(batch, dtype=torch.long, device=device)
                mask_tensor = (sequence_tensor != self.tokenizer.character_to_token(Tokenizer.PAD)).long()

                self.model.train()
                model_output, target = self.model(sequence_tensor, mask_tensor)

                # model_output: (batch, seq, vocab)
                # CrossEntropyLossには (batch, vocab, seq) で渡す
                loss = self.loss_function(model_output.transpose(1, 2), target)

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 0.5)
                self.optimizer.step()
                self.optimizer.zero_grad()

                losses.append(loss.item())

            epoch_loss = float(np.mean(losses))
            loss_per_epoch.append(epoch_loss)

            if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
                print(f"Epoch {epoch:3d} / {epochs}: loss = {epoch_loss:.4f}")

        return loss_per_epoch


def pad_left(sequence: List[int], final_length: int, padding_token: int):
    """
    生成時に、系列長を max_sequence_length に合わせる。
    長すぎるときは末尾だけ残す。
    """
    sequence = sequence[-final_length:]
    return [padding_token] * (final_length - len(sequence)) + sequence


class Generator:
    def __init__(self, model: AutoregressiveWrapper, tokenizer: Tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def generate(
        self,
        max_tokens_to_generate: int,
        prompt: str = "",
        temperature: float = 0.8,
        padding_token: int = None
    ):
        if padding_token is None:
            padding_token = self.tokenizer.character_to_token(Tokenizer.PAD)

        self.model.eval()

        with torch.no_grad():
            start_tokens = self.tokenizer.tokenize(prompt) if prompt else []
            input_tensor = torch.tensor(
                pad_left(
                    sequence=start_tokens,
                    final_length=self.model.max_sequence_length,
                    padding_token=padding_token
                ),
                dtype=torch.long,
                device=device
            )[None, :]

            out = input_tensor

            for _ in range(max_tokens_to_generate):
                x = out[:, -self.model.max_sequence_length:]
                mask = (x != padding_token).long()

                next_token_probabilities = self.model.next_token_probabilities(
                    x=x,
                    mask=mask,
                    temperature=temperature
                )

                next_token = torch.multinomial(next_token_probabilities, num_samples=1)
                out = torch.cat([out, next_token], dim=1)

            generated_tokens = out[0].tolist()
            return self.tokenizer.decode(generated_tokens, skip_special=True)


# ===== 実験用データ =====
# 英語の短文を少量だけ学習させます。
# 本格的なLLMではなく、「仕組みを見るためのミニチュア」です。

training_data = ". ".join([
    "cats rule the world",
    "dogs are the best",
    "elephants have long trunks",
    "monkeys like bananas",
    "pandas eat bamboo",
    "tigers are dangerous",
    "zebras have stripes",
    "lions are the kings of the savannah",
    "giraffes have long necks",
    "hippos are big and scary",
    "rhinos have horns",
    "penguins live in the arctic",
    "polar bears are white"
])

tokenizer = Tokenizer(training_data)

max_sequence_length = 20
embedding_dimension = 64
number_of_heads = 4
number_of_layers = 1
dropout_rate = 0.1

number_of_tokens = tokenizer.size()
print("vocab size =", number_of_tokens)
print("tokens =", tokenizer.dictionary)

model = AutoregressiveWrapper(LanguageModel(
    number_of_tokens=number_of_tokens,
    max_sequence_length=max_sequence_length,
    embedding_dimension=embedding_dimension,
    number_of_heads=number_of_heads,
    number_of_layers=number_of_layers,
    dropout_rate=dropout_rate
)).to(device)

tokenized_training_data = tokenize_and_pad_training_data(
    max_sequence_length=max_sequence_length,
    tokenizer=tokenizer,
    training_data=training_data
)

sequences = create_training_sequences(max_sequence_length, tokenized_training_data)
print("number of training sequences =", len(sequences))
print("example sequence ids =", sequences[0])
print("example sequence text =", tokenizer.decode(sequences[0]))


# ===== 学習 =====
# CPUでも動くように小さめです。
# ColabのGPUなら epochs=100〜200 に増やすと、よりそれらしい文になります。

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
trainer = Trainer(model, tokenizer, optimizer)

loss_per_epoch = trainer.train(
    data=sequences,
    epochs=60,
    batch_size=64
)

plt.figure()
plt.plot(loss_per_epoch)
plt.yscale("log")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.title("Training loss")
plt.show()


# ===== 文章生成 =====
# temperatureを下げると堅く、高くするとランダムになります。

generator = Generator(model, tokenizer)

for prompt in ["elephants", "dogs", "cats", "pandas"]:
    generated_text = generator.generate(
        max_tokens_to_generate=60,
        prompt=prompt,
        temperature=0.8
    )
    print("=" * 60)
    print("prompt:", prompt)
    print(generated_text)


# ===== 保存 =====
# Colab左側のファイル欄から trained_mini_transformer.pt をダウンロードできます。

model.save_checkpoint("trained_mini_transformer.pt")


# - `MaskedSelfAttention` は、現在までの文字だけを見て、次の文字を予測します。
# - `MaskedMultiHeadedSelfAttention` は、複数の見方で文脈を読む仕組みです。
# - `DecoderLayer` を積むと、GPT型の小型Transformerになります。
# - このプログラムでは、語彙が文字単位なので、生成結果は「英単語を理解している」というより、「文字列パターンを学習している」と説明すると正確です。

SyntaxError: invalid character '＃' (U+FF03) (2644910430.py, line 133)